In [10]:
# eeg_gan_final_fixed.py
import torch
import torch.nn as nn
import torch.nn.functional as F
import h5py
from torch.utils.data import Dataset, DataLoader, random_split
from transformers import AutoTokenizer, AutoModel
from torch.optim import Adam
from tqdm.auto import tqdm
import time
from evaluate import load as eval_load

# CONFIG
H5_FILE_PATH = "/home/poorna/data/eeg_dataset_with_qwen.h5"
LOCAL_MODEL_PATH = "/home/poorna/models/bert-base-uncased"
MODEL_SAVE_PATH_G = "eeg_gan_generator.pt"
MODEL_SAVE_PATH_D = "eeg_gan_discriminator.pt"
BATCH_SIZE = 32
EPOCHS = 100
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
LATENT_DIM = 256
SEQ_LEN = 64
VOCAB_SIZE = 32000
LAMBDA_GP = 10
N_CRITIC = 5

# DATASET——————————————————————————————————————
class EEGTextDataset(Dataset):
    def __init__(self, h5_path):
        self.h5 = h5py.File(h5_path, 'r')
        self.eeg = self.h5['eeg']
        self.text = self.h5['input_ids']
        print(f"Loaded {len(self.eeg)} samples")

    def __len__(self): return len(self.eeg)
    def __getitem__(self, idx):
        eeg = torch.from_numpy(self.eeg[idx].astype('float32'))
        text = torch.from_numpy(self.text[idx].astype('int64'))
        return eeg, text

def collate_fn(batch):
    eeg, text = zip(*batch)
    eeg = torch.stack(eeg)
    text = torch.nn.utils.rnn.pad_sequence(text, batch_first=True, padding_value=0)
    return eeg, text

# RESIDUAL LSTM——————————————————————————————
class ResidualLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.res_proj = nn.Linear(input_dim, hidden_dim * 2) if input_dim != hidden_dim * 2 else None
    def forward(self, x):
        residual = x
        out, _ = self.lstm(x)
        if self.res_proj:
            residual = self.res_proj(residual)
        return out + residual

class EEGGenerator(nn.Module):
    def __init__(self, latent_dim=256, vocab_size=32000, seq_len=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, (3,5), (2,2), (1,2)),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, (3,5), (2,2), (1,2)),
            nn.BatchNorm2d(128), nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, latent_dim))
        self.lstm = ResidualLSTM(latent_dim, 512)
        self.out = nn.Linear(1024, vocab_size)
        self.seq_len = seq_len

    def forward(self, eeg):
        x = eeg.unsqueeze(1)
        x = self.conv(x)
        x = self.pool(x)
        x = x.squeeze(-2)          # (B, 128, 256)
        x = x.mean(dim=1)          # (B, 256)
        x = x.unsqueeze(1).repeat(1, self.seq_len, 1)
        x = self.lstm(x)
        return self.out(x)

# DISCRIMINATOR——————————————————————————————
class TextDiscriminator(nn.Module):
    def __init__(self, model_name=LOCAL_MODEL_PATH):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        for param in list(self.bert.parameters())[:-10]:
            param.requires_grad = False
        self.cls = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        return self.cls(outputs.pooler_output).squeeze(-1)

# GRADIENT PENALTY———————————————————————————
def gradient_penalty(D, real_text, fake_text, real_mask, fake_mask):
    B = real_text.size(0)
    alpha = torch.rand(B, 1, 1, device=DEVICE).expand_as(real_text)
    interpolates = (alpha * real_text.float() + (1 - alpha) * fake_text.float()).requires_grad_(True)
    interpolates = torch.clamp(interpolates, 0, VOCAB_SIZE - 1)  # ← CRITICAL
    mask_interp = (interpolates != 0).long()
    d_interp = D(interpolates.long(), mask_interp)
    gradients = torch.autograd.grad(
        outputs=d_interp,
        inputs=interpolates,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True,
        retain_graph=True
    )[0]
    gradients = gradients.view(B, -1)
    return ((gradients.norm(2, dim=1) - 1) ** 2).mean()

In [6]:
# TEST GENERATOR
G = EEGGenerator().to('cpu')
eeg = torch.randn(2, 62, 400)
logits = G(eeg)
print(logits.shape)  # Should print: torch.Size([2, 64, 32000])

torch.Size([2, 64, 32000])


In [12]:
if __name__ == "__main__":
    print("EEG-GAN-TEXT: FINAL LSTM + WGAN-GP")
    print("="*60)

    ds = EEGTextDataset(H5_FILE_PATH)
    tr, te = random_split(ds, [int(0.9*len(ds)), len(ds)-int(0.9*len(ds))],
                          generator=torch.Generator().manual_seed(42))
    trL = DataLoader(tr, BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    teL = DataLoader(te, BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    tokenizer = AutoTokenizer.from_pretrained(LOCAL_MODEL_PATH)
    VOCAB_SIZE = tokenizer.vocab_size  # ← 30522

    G = EEGGenerator(LATENT_DIM, VOCAB_SIZE, SEQ_LEN)  # ← Pass VOCAB_SIZE
    D = TextDiscriminator()
    G = G.to(DEVICE)
    D = D.to(DEVICE)

    opt_G = Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))
    opt_D = Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))

    for epoch in range(1, EPOCHS + 1):
        G.train(); D.train()
        d_loss_total = g_loss_total = 0
        prog = tqdm(trL, desc=f"Epoch {epoch:02d}")
        for eeg, real_text in prog:
            eeg, real_text = eeg.to(DEVICE), real_text.to(DEVICE)

            real_text = torch.clamp(real_text, 0, VOCAB_SIZE - 1)
            real_mask = (real_text != 0).long()

            for _ in range(N_CRITIC):
                opt_D.zero_grad()
                real_score = D(real_text, real_mask)
                fake_logits = G(eeg)
                fake_text = fake_logits.argmax(-1).detach()
                fake_text = torch.clamp(fake_text, 0, VOCAB_SIZE - 1)
                fake_mask = (fake_text != 0).long()
                fake_score = D(fake_text, fake_mask)
                gp = gradient_penalty(D, real_text, fake_text, real_mask, fake_mask)
                d_loss = -(real_score.mean() - fake_score.mean()) + LAMBDA_GP * gp
                d_loss.backward()
                opt_D.step()

            opt_G.zero_grad()
            fake_logits = G(eeg)
            fake_text = fake_logits.argmax(-1)
            fake_text = torch.clamp(fake_text, 0, VOCAB_SIZE - 1)
            fake_mask = (fake_text != 0).long()
            fake_score = D(fake_text, fake_mask)
            g_loss = -fake_score.mean()
            g_loss.backward()
            opt_G.step()

            d_loss_total += d_loss.item()
            g_loss_total += g_loss.item()

        print(f"Epoch {epoch:02d} | D Loss: {d_loss_total/len(trL):.4f} | G Loss: {g_loss_total/len(trL):.4f}")
        if epoch % 20 == 0:
            torch.save(G.state_dict(), MODEL_SAVE_PATH_G)
            torch.save(D.state_dict(), MODEL_SAVE_PATH_D)
            print(f" [Models saved at epoch {epoch}]")

EEG-GAN-TEXT: FINAL LSTM + WGAN-GP
Loaded 28000 samples


RuntimeError: CUDA error: device-side assert triggered
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
 # -------------------------------------------------
    # FINAL EVALUATION
    # -------------------------------------------------
    print("\n" + "="*60)
    print("FINAL EVALUATION ON TEST SET")
    print("="*60)

    G.load_state_dict(torch.load(MODEL_SAVE_PATH_G))
    G.eval()
    bleu_metric = eval_load("bleu")

    predictions = []
    references = []

    print("Generating text from EEG...")
    start_time = time.time()
    with torch.no_grad():
        for eeg, txt in tqdm(teL):
            eeg = eeg.to(DEVICE)
            logits = G(eeg)
            pred_ids = logits.argmax(-1)
            for i in range(eeg.size(0)):
                pred = tokenizer.decode(pred_ids[i].tolist(), skip_special_tokens=True)
                ref = tokenizer.decode(txt[i].tolist(), skip_special_tokens=True)
                predictions.append(pred)
                references.append([ref])
            if len(predictions) >= 200:
                break

    bleu = bleu_metric.compute(predictions=predictions, references=references, max_order=4)['bleu']
    end_time = time.time()
    elapsed = end_time - start_time
    formatted_time = f"{int(elapsed // 60):02d}m {int(elapsed % 60):02d}s"

    print("\n" + "="*60)
    print("EVALUATION RESULTS")
    print(f"Time taken      : {formatted_time}")
    print(f"BLEU-4 Score    : {bleu:.4f}")
    print(f"Samples         : {len(predictions)}")
    print("="*60)

    print("\nSAMPLE GENERATIONS:")
    for i in range(min(3, len(predictions))):
        print(f"Pred: {predictions[i]}")
        print(f"Ref : {references[i][0]}\n")

In [4]:
import torch
import torch.nn as nn

class ResidualLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers,
                            batch_first=True, bidirectional=True, dropout=dropout)
        self.res_proj = nn.Linear(input_dim, hidden_dim * 2) if input_dim != hidden_dim * 2 else None
    def forward(self, x):
        residual = x
        out, _ = self.lstm(x)
        if self.res_proj:
            residual = self.res_proj(residual)
        return out + residual

class EEGGenerator(nn.Module):
    def __init__(self, latent_dim=256, vocab_size=32000, seq_len=64):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 64, (3,5), (2,2), (1,2)),
            nn.BatchNorm2d(64), nn.ReLU(),
            nn.Conv2d(64, 128, (3,5), (2,2), (1,2)),
            nn.BatchNorm2d(128), nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool2d((1, latent_dim))
        self.lstm = ResidualLSTM(latent_dim, 512)
        self.out = nn.Linear(1024, vocab_size)
        self.seq_len = seq_len

    def forward(self, eeg):
        x = eeg.unsqueeze(1)
        x = self.conv(x)
        x = self.pool(x)
        x = x.squeeze(-2)          # (B, 128, 256)
        x = x.mean(dim=1)          # (B, 256)
        x = x.unsqueeze(1).repeat(1, self.seq_len, 1)
        x = self.lstm(x)
        return self.out(x)

# TEST
G = EEGGenerator()
eeg = torch.randn(2, 62, 400)
out = G(eeg)
print("SUCCESS:", out.shape)

SUCCESS: torch.Size([2, 64, 32000])
